# 8장. 더 빠르게: 실제 LLM처럼 계산하기

7장의 GPT와 **계산 결과는 똑같고 속도만 빠른** GPT로 바꿔 끼워요. 이 장은 새 코드를 만들기보다 **실행하고 확인하는 장**입니다. 9장부터는 이 빠른 GPT로 학습해요.

결과가 같은지 확인하는 부분(8.2~8.3)은 CPU로도 되지만, **속도를 재는 부분(8.1, 8.4)은 T4 GPU**에서 실행해야 의미가 있습니다. (런타임 유형: T4 GPU)


### 준비
7장의 기본 GPT(`TinyGPT`)와 패키지의 빠른 GPT(`GPT`), 그리고 시간을 재는 `time_steps`를 불러옵니다.


In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm

import torch
from modu_llm import Vocab, GPTConfig, TinyGPT, GPT, copy_basic_to_fast
from modu_llm import load_stories, encode_texts, get_batch, time_steps

torch.manual_seed(42)
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
vocab = Vocab.build()
cfg = GPTConfig()
stories = load_stories(2000)
data = encode_texts(stories, vocab)
print("장치:", device)


## 8.1 우리 코드는 왜 느릴까

7장 GPT로 학습 몇 단계에 걸리는 시간을 재어 봅니다. 느린 이유는 두 가지예요.

1. 헤드 4개를 **반복문으로 하나씩** 계산합니다. GPU는 큰 계산 하나를 한꺼번에 할 때 가장 빨라요.
2. 모든 숫자를 32비트로 계산합니다. T4는 16비트 계산을 훨씬 빠르게 하는 전용 장치를 갖고 있어요.


In [ ]:
basic = TinyGPT(cfg).to(device)
if device == "cuda":
    n = 10
else:
    n = 2
t_basic = time_steps(basic, data, n=n)
print(f"기본 GPT: 한 단계 {t_basic:.0f}ms")


## 8.2 헤드를 한 번에 계산하는 GPT

패키지의 `GPT`는 7장의 `TinyGPT`와 구조도 부품 이름도 같아요. 다른 곳은 블록 안의 어텐션 하나뿐입니다.
헤드 4개를 따로 계산하는 대신, **헤드를 배치처럼 한 차원으로 세워** 한꺼번에 계산하는 `CausalSelfAttention`을 씁니다. 코드를 꺼내 봅시다.


In [ ]:
import inspect
from modu_llm import CausalSelfAttention

print(inspect.getsource(CausalSelfAttention))


## 8.3 결과가 똑같은지 확인하기

빨라져도 결과가 달라지면 안 돼요. 기본 GPT의 가중치를 빠른 GPT로 옮기고, 같은 입력을 넣어 결과를 비교합니다.

**핵심 코드**: `torch.allclose(out_basic, out_fast, atol=1e-4)`


In [ ]:
basic = basic.cpu().eval()
fast = GPT(cfg)
copy_basic_to_fast(basic, fast)
fast.eval()

xb, yb = get_batch(data, cfg.block_size, 2)
with torch.no_grad():
    out_basic, _ = basic(xb)
    out_fast, _ = fast(xb)

same = torch.allclose(out_basic, out_fast, atol=1e-4)
print("두 GPT의 결과가 같은가요?", same)


이번에는 **반정밀도(16비트)**를 켰을 때와 껐을 때를 비교합니다. GPU가 없으면 둘 다 32비트로 계산돼요.


In [ ]:
amp = device == "cuda"
model = GPT(cfg).to(device)
model.eval()
x, y = get_batch(data, cfg.block_size, 4, device)

with torch.no_grad():
    logits32, loss32 = model(x, y)
    with torch.autocast(device_type=device, dtype=torch.float16, enabled=amp):
        logits16, loss16 = model(x, y)

print("끄고 계산한 점수의 자료형:", logits32.dtype)
print("켜고 계산한 점수의 자료형:", logits16.dtype)
print(f"끄고 계산한 손실: {loss32.item():.4f}")
print(f"켜고 계산한 손실: {loss16.item():.4f}")


## 8.4 얼마나 빨라졌나 재어 보기

세 가지를 비교합니다. GPU가 없으면 반정밀도는 건너뛰어요.


In [ ]:
t_fast = time_steps(GPT(cfg).to(device), data, n=n)

names = ["기본 GPT", "빠른 GPT"]
times = [t_basic, t_fast]
if device == "cuda":
    t_amp = time_steps(GPT(cfg).to(device), data, n=n, amp=True)
    names.append("빠른 GPT + 반정밀도")
    times.append(t_amp)

for i in range(len(names)):
    t = times[i]
    speedup = t_basic / t
    steps = round(30 * 60 * 1000 / t)
    print(f"{names[i]}: 한 단계 {t:.0f}ms, {speedup:.1f}배, 30분에 약 {steps:,}단계")


## 놀이터

본 모델을 건드리지 않는 활동입니다. 건너뛰어도 다음 장에 영향이 없습니다.

배치 크기를 바꾸면 한 단계 시간과 **1초에 읽는 글자 수**가 어떻게 달라지는지 보세요. GPU는 일을 한꺼번에 많이 줄수록 효율이 좋아집니다.


In [ ]:
for bs in [8, 16, 32, 64]:
    model = GPT(cfg).to(device)
    t = time_steps(model, data, n=n, amp=amp, batch_size=bs)
    chars = round(bs * cfg.block_size / t * 1000)
    print(f"배치 {bs}: 한 단계 {t:.0f}ms, 1초에 {chars:,}글자")


## 실제 LLM에서는

- 실제 LLM은 이 장의 기법에 더해, 16비트 중에서도 범위가 넓은 bfloat16, 더 빠른 어텐션 구현(FlashAttention), 모델을 기계어에 가깝게 바꾸는 컴파일(`torch.compile`), 수천 개 GPU에 나눠 계산하기를 함께 씁니다.
- 원리를 배울 때는 7장처럼 쉬운 코드로, 실제로 학습할 때는 이 장처럼 빠른 코드로. 실제 LLM 개발도 같은 길을 걸어요.
